In [2]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/finqa-rag-lora-ablation

Mounted at /content/drive
/content/drive/MyDrive/finqa-rag-lora-ablation


In [3]:
!pip install datasets transformers -q

In [4]:
import os
import json
import urllib.request

DATA_URLS = {
    'train': 'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/train.json',
    'dev':   'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/dev.json',
    'test':  'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/test.json',
}

RAW_DIR = 'data/raw'
os.makedirs(RAW_DIR, exist_ok=True)

for split, url in DATA_URLS.items():
    out_path = f'{RAW_DIR}/{split}.json'
    if os.path.exists(out_path):
        print(f" {split}.json exist")
        continue
    print(f" download {split}.json ...")
    urllib.request.urlretrieve(url, out_path)
    print(f"save {out_path}")

finqa = {}
for split in ['train', 'dev', 'test']:
    with open(f'{RAW_DIR}/{split}.json', 'r') as f:
        finqa[split] = json.load(f)
    print(f"{split}: {len(finqa[split])} ")

 train.json exist
 dev.json exist
 test.json exist
train: 6251 
dev: 883 
test: 1147 


In [5]:
sample = finqa['train'][0]

print("top:", list(sample.keys()))
print()
print("=" * 60)

import json
text = json.dumps(sample, indent=2, ensure_ascii=False)
print(text[:3000])
print("..." if len(text) > 3000 else "")

top: ['pre_text', 'post_text', 'filename', 'table_ori', 'table', 'qa', 'id', 'table_retrieved', 'text_retrieved', 'table_retrieved_all', 'text_retrieved_all']

{
  "pre_text": [
    "interest rate to a variable interest rate based on the three-month libor plus 2.05% ( 2.05 % ) ( 2.34% ( 2.34 % ) as of october 31 , 2009 ) .",
    "if libor changes by 100 basis points , our annual interest expense would change by $ 3.8 million .",
    "foreign currency exposure as more fully described in note 2i .",
    "in the notes to consolidated financial statements contained in item 8 of this annual report on form 10-k , we regularly hedge our non-u.s .",
    "dollar-based exposures by entering into forward foreign currency exchange contracts .",
    "the terms of these contracts are for periods matching the duration of the underlying exposure and generally range from one month to twelve months .",
    "currently , our largest foreign currency exposure is the euro , primarily because our european op

In [6]:
for i in range(5):
    s = finqa['train'][i]
    qa = s.get('qa', {})
    print(f"--- Sample {i} ---")
    print(f"Question: {qa.get('question', 'N/A')}")
    print(f"Answer:   {qa.get('answer', 'N/A')}")
    print(f"Program:  {qa.get('program', 'N/A')}")
    print()

--- Sample 0 ---
Question: what is the the interest expense in 2009?
Answer:   380
Program:  divide(100, 100), divide(3.8, #0)

--- Sample 1 ---
Question: during the 2012 year , did the equity awards in which the prescribed performance milestones were achieved exceed the equity award compensation expense for equity granted during the year?
Answer:   
Program:  multiply(607, 18.13), multiply(#0, const_1000), multiply(3.3, const_1000000), greater(#1, #2)

--- Sample 2 ---
Question: what was the total operating expenses in 2018 in millions
Answer:   41932
Program:  divide(9896, 23.6%)

--- Sample 3 ---
Question: what percentage of total cash and investments as of dec . 29 2012 was comprised of available-for-sale investments?
Answer:   53%
Program:  divide(14001, 26302)

--- Sample 4 ---
Question: what is the growth rate in net revenue in 2008?
Answer:   -3.2%
Program:  subtract(959.2, 991.1), divide(#0, 991.1)



In [7]:
import numpy as np
def get_context_length(sample):
    pre = ' '.join(sample.get('pre_text', []))
    post = ' '.join(sample.get('post_text', []))
    return len((pre + ' ' + post).split())

lengths = [get_context_length(s) for s in finqa['train']]

print(f"Context Word Count Statistics:")
print(f"  Mean: {np.mean(lengths):.0f} words")
print(f"  Median: {np.median(lengths):.0f} words")
print(f"  Min/Max: {np.min(lengths)} / {np.max(lengths)} words")

Context Word Count Statistics:
  Mean: 632 words
  Median: 622 words
  Min/Max: 12 / 2618 words


In [8]:
print("First 20 answer samples:")
for i in range(20):
    ans = finqa['train'][i]['qa'].get('answer', '')
    print(f"  [{i}] {ans!r}")

First 20 answer samples:
  [0] '380'
  [1] ''
  [2] '41932'
  [3] '53%'
  [4] '-3.2%'
  [5] '56.25%'
  [6] '7.4'
  [7] '63.6%'
  [8] '96.55%'
  [9] '56.6'
  [10] '6.9'
  [11] ''
  [12] '65.3%'
  [13] '0.3%'
  [14] '28%'
  [15] '65.6%'
  [16] '20.2%'
  [17] '12.03%'
  [18] '3.61%'
  [19] '93.4%'


In [9]:
import re

def classify_answer(answer: str, program: str) -> str:
    """Classify samples into percentage / numeric / boolean / other based on answer and program"""
    answer = (answer or '').strip()
    program = (program or '').strip()

    # boolean: program ends with a comparison operation
    boolean_ops = ['greater', 'less', 'equal', 'smaller', 'greater_or_equal']
    last_op = program.split(',')[-1].strip().split('(')[0] if program else ''
    if last_op in boolean_ops:
        return 'boolean'

    # percentage: answer contains %
    if '%' in answer:
        return 'percentage'

    # numeric: answer is a number (may include negative signs, decimals, or commas)
    if re.match(r'^-?[\d,]+\.?\d*$', answer):
        return 'numeric'

    # other: includes empty strings or text-based answers
    return 'other'


def classify_complexity(program: str) -> str:
    """Classify as simple / complex based on the number of steps in the program"""
    if not program:
        return 'simple'
    n_ops = len([op for op in program.split(',') if op.strip()])
    return 'simple' if n_ops <= 2 else 'complex'


# Test the classification functions
test_cases = [
    ('380', 'divide(100, 100), divide(3.8, #0)'),
    ('53%', 'divide(14001, 26302)'),
    ('', 'multiply(607, 18.13), multiply(#0, const_1000), multiply(3.3, const_1000000), greater(#1, #2)'),
]
for ans, prog in test_cases:
    print(f"answer={ans!r:10s} -> {classify_answer(ans, prog):10s} | "
          f"complexity={classify_complexity(prog)}")

answer='380'      -> numeric    | complexity=complex
answer='53%'      -> percentage | complexity=simple
answer=''         -> other      | complexity=complex


In [10]:
from collections import Counter

# Calculate distributions on the train set
ans_types = []
complexities = []
combined = []

for s in finqa['train']:
    qa = s.get('qa', {})
    ans = qa.get('answer', '')
    prog = qa.get('program', '')

    at = classify_answer(ans, prog)
    cx = classify_complexity(prog)

    ans_types.append(at)
    complexities.append(cx)
    combined.append(f"{at}_{cx}")

print("=== Answer Type Distribution ===")
for k, v in Counter(ans_types).most_common():
    print(f"  {k:12s}: {v:5d} ({v/len(ans_types)*100:.1f}%)")

print("\n=== Complexity Distribution ===")
for k, v in Counter(complexities).most_common():
    print(f"  {k:12s}: {v:5d} ({v/len(complexities)*100:.1f}%)")

print("\n=== Joint Distribution (6 strata) ===")
for k, v in sorted(Counter(combined).items()):
    print(f"  {k:25s}: {v:5d} ({v/len(combined)*100:.1f}%)")

=== Answer Type Distribution ===
  percentage  :  3508 (56.1%)
  numeric     :  2440 (39.0%)
  other       :   303 (4.8%)

=== Complexity Distribution ===
  simple      :  3717 (59.5%)
  complex     :  2534 (40.5%)

=== Joint Distribution (6 strata) ===
  numeric_complex          :   675 (10.8%)
  numeric_simple           :  1765 (28.2%)
  other_complex            :    82 (1.3%)
  other_simple             :   221 (3.5%)
  percentage_complex       :  1777 (28.4%)
  percentage_simple        :  1731 (27.7%)


In [11]:
# Inspect samples in the 'other' category to decide whether to include them
print("=== 'other' class samples (top 10) ===")
count = 0
for s in finqa['train']:
    qa = s.get('qa', {})
    ans = qa.get('answer', '')
    prog = qa.get('program', '')
    if classify_answer(ans, prog) == 'other':
        print(f"  answer={ans!r}, program={prog!r}")
        count += 1
        if count >= 10:
            break

=== 'other' class samples (top 10) ===
  answer='', program='multiply(607, 18.13), multiply(#0, const_1000), multiply(3.3, const_1000000), greater(#1, #2)'
  answer='', program='add(15553, 7376)'
  answer='$ 13 million', program='subtract(78.0, 75.3), subtract(58.0, 49.9), subtract(54.0, 51.8), add(#0, #1), add(#2, #3)'
  answer='yes', program='greater(189.57, 137.82)'
  answer='$ 7.6 million', program='subtract(34.6, 24.8)'
  answer='yes', program='greater(5941210, 4852978)'
  answer='$ 4236 million', program='multiply(10086, 42%)'
  answer='', program='subtract(51%, 20%), divide(343, #0)'
  answer='yes', program='greater(45, 25)'
  answer='no', program='greater(1, 10)'


In [12]:
%%writefile data/preprocess.py
"""
FinQA Data Preprocessing Pipeline

Downloads the original FinQA dataset from GitHub, classifies samples by
answer type (percentage/numeric/boolean) and complexity (simple/complex),
performs stratified sampling with intentional oversampling of boolean
questions for reliable error analysis.

Usage:
    python data/preprocess.py
    python data/preprocess.py --n_samples 500 --seed 42

Output:
    data/raw/{train,dev,test}.json     - original FinQA splits
    data/processed/finqa_500.json       - 500 stratified samples
    data/processed/sampling_stats.json  - sampling statistics for the report
    data/samples/finqa_10_demo.json     - 10-sample demo (committed to git)
"""

import argparse
import json
import os
import random
import re
import urllib.request
from collections import Counter, defaultdict
from typing import Dict, List

# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

DATA_URLS = {
    'train': 'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/train.json',
    'dev':   'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/dev.json',
    'test':  'https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/test.json',
}

# Target sample count per stratum (must sum to total N)
STRATUM_QUOTAS = {
    'percentage_simple':  130,
    'percentage_complex': 130,
    'numeric_simple':     130,
    'numeric_complex':     60,
    'boolean_simple':      40,
    'boolean_complex':     10,
}
TOTAL_SAMPLES = sum(STRATUM_QUOTAS.values())  # 500

BOOLEAN_OPS = {'greater', 'less', 'equal', 'smaller',
               'greater_or_equal', 'less_or_equal'}


# -----------------------------------------------------------------------------
# Classification functions
# -----------------------------------------------------------------------------

def classify_answer(answer: str, program: str) -> str:
    """Classify a sample by answer type: percentage / numeric / boolean / other."""
    answer = (answer or '').strip().lower()
    program = (program or '').strip()

    # 1. boolean: explicit yes/no, OR program ends with comparison op
    if answer in ('yes', 'no', 'true', 'false'):
        return 'boolean'
    if program:
        last_op = program.split(',')[-1].strip().split('(')[0].strip()
        if last_op in BOOLEAN_OPS:
            return 'boolean'

    # 2. percentage
    if '%' in answer:
        return 'percentage'

    # 3. numeric (allows $, comma, million/billion suffixes)
    cleaned = answer.replace('$', '').replace(',', '').replace(' ', '')
    cleaned = re.sub(r'(million|billion|thousand|m|bn|k)$', '', cleaned)
    try:
        float(cleaned)
        return 'numeric'
    except (ValueError, TypeError):
        return 'other'


def classify_complexity(program: str) -> str:
    """Simple if program has <=2 ops, complex otherwise."""
    if not program:
        return 'simple'
    n_ops = len([op for op in program.split(',') if op.strip()])
    return 'simple' if n_ops <= 2 else 'complex'


def get_stratum(sample: dict) -> str:
    """Return the stratum label for a sample, e.g. 'percentage_simple'."""
    qa = sample.get('qa', {}) or {}
    ans_type = classify_answer(qa.get('answer', ''), qa.get('program', ''))
    complexity = classify_complexity(qa.get('program', ''))
    return f"{ans_type}_{complexity}"


# -----------------------------------------------------------------------------
# Data loading
# -----------------------------------------------------------------------------

def download_finqa(raw_dir: str = 'data/raw') -> Dict[str, list]:
    """Download FinQA splits from the official GitHub repo if not already present."""
    os.makedirs(raw_dir, exist_ok=True)
    data = {}
    for split, url in DATA_URLS.items():
        path = os.path.join(raw_dir, f'{split}.json')
        if not os.path.exists(path):
            print(f"Downloading {split}.json ...")
            urllib.request.urlretrieve(url, path)
        with open(path, 'r') as f:
            data[split] = json.load(f)
        print(f"  {split}: {len(data[split])} samples")
    return data


# -----------------------------------------------------------------------------
# Stratified sampling
# -----------------------------------------------------------------------------

def stratified_sample(samples: List[dict], quotas: Dict[str, int],
                      seed: int = 42) -> Dict[str, list]:
    """
    Bucket samples by stratum, then randomly draw `quotas[stratum]` from each.
    Returns dict mapping stratum -> list of sampled records.
    Raises if any stratum has fewer than the requested quota.
    """
    rng = random.Random(seed)

    buckets = defaultdict(list)
    for s in samples:
        stratum = get_stratum(s)
        if stratum in quotas:
            buckets[stratum].append(s)

    sampled = {}
    for stratum, quota in quotas.items():
        pool = buckets.get(stratum, [])
        if len(pool) < quota:
            raise ValueError(
                f"Stratum {stratum!r} has only {len(pool)} samples but "
                f"{quota} were requested. Adjust STRATUM_QUOTAS."
            )
        sampled[stratum] = rng.sample(pool, quota)
        print(f"  {stratum:25s}: drew {quota:3d} from pool of {len(pool):4d}")

    return sampled


# -----------------------------------------------------------------------------
# Main
# -----------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--seed', type=int, default=42,
                        help='Random seed for reproducibility')
    parser.add_argument('--raw_dir', default='data/raw')
    parser.add_argument('--out_dir', default='data/processed')
    parser.add_argument('--demo_dir', default='data/samples')
    args = parser.parse_args()

    print("=" * 60)
    print("Step 1: Download FinQA from GitHub")
    print("=" * 60)
    finqa = download_finqa(args.raw_dir)

    print("\n" + "=" * 60)
    print("Step 2: Compute full distribution on train split")
    print("=" * 60)
    full_strata = Counter(get_stratum(s) for s in finqa['train'])
    total = sum(full_strata.values())
    natural_weights = {}
    for stratum in STRATUM_QUOTAS:
        n = full_strata.get(stratum, 0)
        natural_weights[stratum] = n / total if total > 0 else 0
        print(f"  {stratum:25s}: {n:5d} ({natural_weights[stratum]*100:.1f}%)")

    print("\n" + "=" * 60)
    print(f"Step 3: Stratified sample {TOTAL_SAMPLES} from train")
    print("=" * 60)
    sampled_by_stratum = stratified_sample(finqa['train'], STRATUM_QUOTAS,
                                            seed=args.seed)

    # Flatten to single list, attach stratum metadata for downstream use
    flat_samples = []
    for stratum, items in sampled_by_stratum.items():
        for item in items:
            item['_stratum'] = stratum
            flat_samples.append(item)

    print(f"\n  Total sampled: {len(flat_samples)}")

    print("\n" + "=" * 60)
    print("Step 4: Save outputs")
    print("=" * 60)
    os.makedirs(args.out_dir, exist_ok=True)
    os.makedirs(args.demo_dir, exist_ok=True)

    # Main processed file (NOT committed, in .gitignore)
    out_path = os.path.join(args.out_dir, f'finqa_{TOTAL_SAMPLES}.json')
    with open(out_path, 'w') as f:
        json.dump(flat_samples, f, indent=2)
    print(f"  Saved: {out_path}")

    # Sampling statistics (committed, useful for the report)
    stats = {
        'total_samples': TOTAL_SAMPLES,
        'random_seed': args.seed,
        'stratum_quotas': STRATUM_QUOTAS,
        'natural_weights': natural_weights,
        'note': 'Boolean strata are intentionally oversampled '
                '(~10% sampled vs ~3% natural) for reliable error analysis. '
                'Use natural_weights to recompute population-weighted metrics.'
    }
    stats_path = os.path.join(args.out_dir, 'sampling_stats.json')
    with open(stats_path, 'w') as f:
        json.dump(stats, f, indent=2)
    print(f"  Saved: {stats_path}")

    # Demo file with 10 samples (committed for TA reproducibility)
    demo = flat_samples[:10]
    demo_path = os.path.join(args.demo_dir, 'finqa_10_demo.json')
    with open(demo_path, 'w') as f:
        json.dump(demo, f, indent=2)
    print(f"  Saved: {demo_path}")

    print("\n✅ Done.")


if __name__ == '__main__':
    main()

Overwriting data/preprocess.py


In [13]:
%cd /content/drive/MyDrive/finqa-rag-lora-ablation
!python data/preprocess.py

/content/drive/MyDrive/finqa-rag-lora-ablation
Step 1: Download FinQA from GitHub
  train: 6251 samples
  dev: 883 samples
  test: 1147 samples

Step 2: Compute full distribution on train split
  percentage_simple        :  1731 (27.7%)
  percentage_complex       :  1777 (28.4%)
  numeric_simple           :  1830 (29.3%)
  numeric_complex          :   700 (11.2%)
  boolean_simple           :   107 (1.7%)
  boolean_complex          :    13 (0.2%)

Step 3: Stratified sample 500 from train
  percentage_simple        : drew 130 from pool of 1731
  percentage_complex       : drew 130 from pool of 1777
  numeric_simple           : drew 130 from pool of 1830
  numeric_complex          : drew  60 from pool of  700
  boolean_simple           : drew  40 from pool of  107
  boolean_complex          : drew  10 from pool of   13

  Total sampled: 500

Step 4: Save outputs
  Saved: data/processed/finqa_500.json
  Saved: data/processed/sampling_stats.json
  Saved: data/samples/finqa_10_demo.json

✅ D

In [14]:
import json
from collections import Counter

# Inspect the main output file
with open('data/processed/finqa_500.json') as f:
    data = json.load(f)
print(f"Main file: {len(data)} samples")
print(f"First sample stratum: {data[0]['_stratum']}")
print(f"First sample fields: {list(data[0].keys())[:5]}...")  # Only view the first 5

# Check the sampling statistics
with open('data/processed/sampling_stats.json') as f:
    stats = json.load(f)
print("\nSampling statistics:")
print(json.dumps(stats, indent=2))

print("\nActual count per stratum:")
strata_counts = Counter(s['_stratum'] for s in data)
for k, v in sorted(strata_counts.items()):
    print(f"  {k:25s}: {v}")

Main file: 500 samples
First sample stratum: percentage_simple
First sample fields: ['pre_text', 'post_text', 'filename', 'table_ori', 'table']...

Sampling statistics:
{
  "total_samples": 500,
  "random_seed": 42,
  "stratum_quotas": {
    "percentage_simple": 130,
    "percentage_complex": 130,
    "numeric_simple": 130,
    "numeric_complex": 60,
    "boolean_simple": 40,
    "boolean_complex": 10
  },
  "natural_weights": {
    "percentage_simple": 0.27691569348904177,
    "percentage_complex": 0.2842745160774276,
    "numeric_simple": 0.2927531594944809,
    "numeric_complex": 0.11198208286674133,
    "boolean_simple": 0.017117261238201887,
    "boolean_complex": 0.002079667253239482
  },
  "note": "Boolean strata are intentionally oversampled (~10% sampled vs ~3% natural) for reliable error analysis. Use natural_weights to recompute population-weighted metrics."
}

Actual count per stratum:
  boolean_complex          : 10
  boolean_simple           : 40
  numeric_complex        

In [15]:
%cd /content/drive/MyDrive/finqa-rag-lora-ablation

gitignore_additions = """
# === FinQA project specific ===
# Raw data downloaded from GitHub (regenerate with: python data/preprocess.py)
data/raw/
data/processed/

# Model checkpoints (saved to Google Drive instead)
checkpoints/
*.bin
*.safetensors

# Wandb logs
wandb/

# Jupyter cache
.ipynb_checkpoints/
"""

with open('.gitignore', 'a') as f:
    f.write(gitignore_additions)

print("Updated .gitignore. Last 20 lines:")
!tail -20 .gitignore

/content/drive/MyDrive/finqa-rag-lora-ablation
Updated .gitignore. Last 20 lines:
wandb/

# Jupyter cache
.ipynb_checkpoints/

# === FinQA project specific ===
# Raw data downloaded from GitHub (regenerate with: python data/preprocess.py)
data/raw/
data/processed/

# Model checkpoints (saved to Google Drive instead)
checkpoints/
*.bin
*.safetensors

# Wandb logs
wandb/

# Jupyter cache
.ipynb_checkpoints/


In [16]:
!git status

Refresh index: 100% (18/18), done.
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore
	modified:   notebooks/01_data_exploration.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	results/metrics/c1_metrics_20260504_114932.json
	results/metrics/c1_predictions_20260504_114932.json
	src/__init__.py
	src/evaluation/__init__.py
	src/evaluation/metrics.py
	src/models/__init__.py
	src/models/baseline.py
	src/pipelines/__init__.py
	src/pipelines/c1_baseline.py
	src/retrieval/__init__.py
	src/utils/__init__.py
	src/utils/data_utils.py

no changes added to commit (use "git add" and/or "git commit -a")


In [17]:
%cd /content/drive/MyDrive/finqa-rag-lora-ablation

!ls -la *.ipynb 2>/dev/null
!ls -la notebooks/

/content/drive/MyDrive/finqa-rag-lora-ablation
total 50
-rw------- 1 root root 50877 May  4 11:50 01_data_exploration.ipynb


In [18]:
!mv 01_data_exploration.ipynb notebooks/01_data_exploration.ipynb

!ls notebooks/

mv: cannot stat '01_data_exploration.ipynb': No such file or directory
01_data_exploration.ipynb


In [19]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .gitignore
	modified:   notebooks/01_data_exploration.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	results/metrics/c1_metrics_20260504_114932.json
	results/metrics/c1_predictions_20260504_114932.json
	src/__init__.py
	src/evaluation/__init__.py
	src/evaluation/metrics.py
	src/models/__init__.py
	src/models/baseline.py
	src/pipelines/__init__.py
	src/pipelines/c1_baseline.py
	src/retrieval/__init__.py
	src/utils/__init__.py
	src/utils/data_utils.py

no changes added to commit (use "git add" and/or "git commit -a")


In [20]:
!git add .gitignore
!git add data/preprocess.py
!git add data/samples/finqa_10_demo.json
!git add notebooks/01_data_exploration.ipynb

!git status   #

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   .gitignore
	modified:   notebooks/01_data_exploration.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	results/metrics/c1_metrics_20260504_114932.json
	results/metrics/c1_predictions_20260504_114932.json
	src/__init__.py
	src/evaluation/__init__.py
	src/evaluation/metrics.py
	src/models/__init__.py
	src/models/baseline.py
	src/pipelines/__init__.py
	src/pipelines/c1_baseline.py
	src/retrieval/__init__.py
	src/utils/__init__.py
	src/utils/data_utils.py



In [21]:
%cd /content/drive/MyDrive/finqa-rag-lora-ablation

# 重新设置 git 身份（每次 Colab 重连都要做一次）
!git config --global user.email "zx2536@columbia.edu"
!git config --global user.name "zx2536"

# 验证设置成功
!git config --global user.email
!git config --global user.name

/content/drive/MyDrive/finqa-rag-lora-ablation
zx2536@columbia.edu
zx2536


In [22]:
!git commit -m "Add FinQA preprocessing with stratified sampling, classification by answer type and complexity, 500-sample stratified output, and updated .gitignore"

[main 11a85e9] Add FinQA preprocessing with stratified sampling, classification by answer type and complexity, 500-sample stratified output, and updated .gitignore
 2 files changed, 17 insertions(+), 1 deletion(-)
 rewrite notebooks/01_data_exploration.ipynb (98%)


In [23]:
!git push origin main

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 6.91 KiB | 642.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/zx2536-gif/finqa-rag-lora-ablation.git
   f2eac1f..11a85e9  main -> main


In [24]:
%%writefile src/utils/data_utils.py
"""
Data formatting utilities for FinQA pipelines.

Converts raw FinQA samples (with pre_text, table, post_text) into
prompt-ready strings that can be fed to LLMs.
"""

from typing import List, Dict, Any


def table_to_markdown(table: List[List[str]]) -> str:
    """Convert a 2D table to a markdown-style string.

    Args:
        table: 2D list, e.g. [["", "2018", "2017"], ["Revenue", "100", "90"]]

    Returns:
        Markdown-formatted string with rows separated by newlines.
    """
    if not table or not table[0]:
        return ""
    lines = []
    for row in table:
        # Clean each cell: strip whitespace, replace empty strings
        cells = [str(cell).strip() if cell else "-" for cell in row]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def build_context(sample: Dict[str, Any]) -> str:
    """Build a single context string from a FinQA sample.

    Concatenates pre_text + table (markdown) + post_text into one document.
    """
    pre = " ".join(sample.get('pre_text', []))
    table_md = table_to_markdown(sample.get('table', []))
    post = " ".join(sample.get('post_text', []))

    parts = []
    if pre:
        parts.append(pre)
    if table_md:
        parts.append("Table:\n" + table_md)
    if post:
        parts.append(post)

    return "\n\n".join(parts)


def build_prompt(sample: Dict[str, Any],
                 max_context_words: int = 350) -> str:
    """Build a zero-shot prompt for Flan-T5.

    Truncates context to max_context_words to fit Flan-T5's 512-token limit.
    (~350 words leaves room for question + prompt template + answer.)

    Args:
        sample: FinQA sample dict
        max_context_words: word-level truncation limit

    Returns:
        Complete prompt string ready for tokenization.
    """
    context = build_context(sample)

    # Word-level truncation (rough but safe)
    words = context.split()
    if len(words) > max_context_words:
        context = " ".join(words[:max_context_words]) + " [...]"

    question = sample.get('qa', {}).get('question', '')

    prompt = (
        "Read the following financial document and answer the question. "
        "Give a short, direct answer (a number, percentage, or yes/no).\n\n"
        f"Document:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )
    return prompt


def get_gold_answer(sample: Dict[str, Any]) -> str:
    """Extract the ground-truth answer string."""
    return str(sample.get('qa', {}).get('answer', '')).strip()

Overwriting src/utils/data_utils.py


In [25]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/finqa-rag-lora-ablation')

from src.utils.data_utils import build_prompt, get_gold_answer
import json

with open('data/processed/finqa_500.json') as f:
    samples = json.load(f)

sample = samples[0]
prompt = build_prompt(sample)
gold = get_gold_answer(sample)

print("=== PROMPT ===")
print(prompt[:1500])
print("...")
print("\n=== GOLD ANSWER ===")
print(repr(gold))
print("\n=== Stratum ===")
print(sample['_stratum'])

=== PROMPT ===
Read the following financial document and answer the question. Give a short, direct answer (a number, percentage, or yes/no).

Document:
abiomed , inc . and subsidiaries notes to consolidated financial statements 2014 ( continued ) note 8 . goodwill and in-process research and development ( continued ) the company has no accumulated impairment losses on goodwill . the company performed a step 0 qualitative assessment during the annual impairment review for fiscal 2015 as of october 31 , 2014 and concluded that it is not more likely than not that the fair value of the company 2019s single reporting unit is less than its carrying amount . therefore , the two-step goodwill impairment test for the reporting unit was not necessary in fiscal 2015 . as described in note 3 . 201cacquisitions , 201d in july 2014 , the company acquired ecp and ais and recorded $ 18.5 million of ipr&d . the estimated fair value of the ipr&d was determined using a probability-weighted income approac

In [26]:
%%writefile src/evaluation/metrics.py
"""
Evaluation metrics for FinQA QA tasks.

Implements F1 and ROUGE-L for textual question answering, plus utilities
for aggregating results per stratum.
"""

import re
import string
from collections import Counter
from typing import List, Dict


def normalize_answer(s: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace.

    Standard normalization from SQuAD and downstream QA benchmarks.
    """
    s = (s or "").lower().strip()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def f1_score(prediction: str, gold: str) -> float:
    """Token-level F1 between prediction and gold answer.

    Returns 0.0 if either is empty (after normalization).
    """
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)  # 1.0 if both empty, else 0.0

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def lcs_length(a: List[str], b: List[str]) -> int:
    """Longest Common Subsequence length, used by ROUGE-L."""
    m, n = len(a), len(b)
    if m == 0 or n == 0:
        return 0
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]


def rouge_l(prediction: str, gold: str) -> float:
    """ROUGE-L F-measure between prediction and gold."""
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    if not pred_tokens or not gold_tokens:
        return float(pred_tokens == gold_tokens)

    lcs = lcs_length(pred_tokens, gold_tokens)
    if lcs == 0:
        return 0.0

    precision = lcs / len(pred_tokens)
    recall = lcs / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)


def evaluate_predictions(predictions: List[str], golds: List[str],
                         strata: List[str] = None) -> Dict:
    """Compute aggregate F1 and ROUGE-L scores.

    Args:
        predictions: list of model output strings
        golds: list of ground truth strings
        strata: optional list of stratum labels for per-stratum breakdown

    Returns:
        Dict with 'overall' metrics and optionally 'by_stratum'.
    """
    assert len(predictions) == len(golds), \
        f"Mismatched lengths: {len(predictions)} preds vs {len(golds)} golds"

    f1s = [f1_score(p, g) for p, g in zip(predictions, golds)]
    rouges = [rouge_l(p, g) for p, g in zip(predictions, golds)]

    results = {
        'overall': {
            'n_samples': len(predictions),
            'f1': sum(f1s) / len(f1s),
            'rouge_l': sum(rouges) / len(rouges),
        }
    }

    # Per-stratum breakdown
    if strata is not None:
        assert len(strata) == len(predictions)
        by_stratum = {}
        for stratum in set(strata):
            indices = [i for i, s in enumerate(strata) if s == stratum]
            by_stratum[stratum] = {
                'n_samples': len(indices),
                'f1': sum(f1s[i] for i in indices) / len(indices),
                'rouge_l': sum(rouges[i] for i in indices) / len(indices),
            }
        results['by_stratum'] = by_stratum

    return results

Overwriting src/evaluation/metrics.py


In [27]:
from src.evaluation.metrics import f1_score, rouge_l, evaluate_predictions

# Quick sanity checks
print("Identical:")
print(f"  F1='{f1_score('53%', '53%'):.3f}'  ROUGE-L='{rouge_l('53%', '53%'):.3f}'")

print("Partial overlap:")
print(f"  F1='{f1_score('about 53 percent', '53%'):.3f}'  ROUGE-L='{rouge_l('about 53 percent', '53%'):.3f}'")

print("Wrong answer:")
print(f"  F1='{f1_score('100', '53%'):.3f}'  ROUGE-L='{rouge_l('100', '53%'):.3f}'")

print("\nAggregate test:")
preds = ["53%", "100", "yes"]
golds = ["53%", "200", "no"]
strata = ["percentage_simple", "numeric_simple", "boolean_simple"]
results = evaluate_predictions(preds, golds, strata)
import json
print(json.dumps(results, indent=2))

Identical:
  F1='1.000'  ROUGE-L='1.000'
Partial overlap:
  F1='0.500'  ROUGE-L='0.500'
Wrong answer:
  F1='0.000'  ROUGE-L='0.000'

Aggregate test:
{
  "overall": {
    "n_samples": 3,
    "f1": 0.3333333333333333,
    "rouge_l": 0.3333333333333333
  },
  "by_stratum": {
    "percentage_simple": {
      "n_samples": 1,
      "f1": 1.0,
      "rouge_l": 1.0
    },
    "numeric_simple": {
      "n_samples": 1,
      "f1": 0.0,
      "rouge_l": 0.0
    },
    "boolean_simple": {
      "n_samples": 1,
      "f1": 0.0,
      "rouge_l": 0.0
    }
  }
}


In [28]:
%%writefile src/models/baseline.py
"""
Flan-T5-Base zero-shot baseline model wrapper.

Loads the model once, exposes a `predict_batch` method for inference.
"""

from typing import List
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer


class FlanT5Baseline:
    """Zero-shot inference wrapper for Flan-T5-Base."""

    def __init__(self, model_name: str = "google/flan-t5-base",
                 device: str = None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device

        print(f"Loading {model_name} on {device}...")
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        print(f"Model loaded. Total params: {sum(p.numel() for p in self.model.parameters()):,}")

    @torch.no_grad()
    def predict_batch(self, prompts: List[str],
                      max_input_length: int = 512,
                      max_new_tokens: int = 64,
                      batch_size: int = 8) -> List[str]:
        """Generate predictions for a list of prompts.

        Args:
            prompts: list of input strings
            max_input_length: truncate prompts to this many tokens
            max_new_tokens: max tokens in generated answer
            batch_size: how many prompts to process per batch

        Returns:
            List of decoded prediction strings.
        """
        predictions = []
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i : i + batch_size]
            inputs = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_input_length,
            ).to(self.device)

            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # deterministic
                num_beams=1,
            )

            decoded = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
            predictions.extend(decoded)

        return predictions

Overwriting src/models/baseline.py


In [29]:
%%writefile src/pipelines/c1_baseline.py
"""
C1: Zero-shot baseline pipeline.

Loads stratified FinQA samples, generates predictions using Flan-T5-Base
with no retrieval and no fine-tuning, computes F1 and ROUGE-L metrics.

Usage:
    python -m src.pipelines.c1_baseline
    python -m src.pipelines.c1_baseline --n_samples 50  # quick test
"""

import argparse
import json
import os
import sys
from datetime import datetime

# Allow running as a script
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(
    os.path.abspath(__file__)))))

from src.utils.data_utils import build_prompt, get_gold_answer
from src.models.baseline import FlanT5Baseline
from src.evaluation.metrics import evaluate_predictions


def run_c1(data_path: str, out_dir: str, model_name: str = "google/flan-t5-base",
           n_samples: int = None, batch_size: int = 8):
    """Run the C1 zero-shot baseline pipeline end-to-end."""

    # === Step 1: Load data ===
    print(f"Loading data from {data_path}...")
    with open(data_path) as f:
        samples = json.load(f)
    if n_samples:
        samples = samples[:n_samples]
        print(f"  Subsampled to {n_samples} for quick testing")
    print(f"  Loaded {len(samples)} samples")

    # === Step 2: Build prompts and gold answers ===
    print("\nBuilding prompts...")
    prompts = [build_prompt(s) for s in samples]
    golds = [get_gold_answer(s) for s in samples]
    strata = [s['_stratum'] for s in samples]

    # === Step 3: Load model and run inference ===
    model = FlanT5Baseline(model_name=model_name)
    print(f"\nGenerating predictions (batch_size={batch_size})...")
    predictions = model.predict_batch(prompts, batch_size=batch_size)

    # === Step 4: Evaluate ===
    print("\nEvaluating...")
    results = evaluate_predictions(predictions, golds, strata)

    print(f"\n=== C1 Baseline Results ===")
    print(f"Overall ({results['overall']['n_samples']} samples):")
    print(f"  F1:      {results['overall']['f1']:.4f}")
    print(f"  ROUGE-L: {results['overall']['rouge_l']:.4f}")

    print(f"\nPer-stratum:")
    for stratum, m in sorted(results['by_stratum'].items()):
        print(f"  {stratum:25s}  n={m['n_samples']:3d}  "
              f"F1={m['f1']:.4f}  ROUGE-L={m['rouge_l']:.4f}")

    # === Step 5: Save outputs ===
    os.makedirs(out_dir, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    # Metrics file (committed to git via results/metrics/)
    metrics_path = os.path.join(out_dir, f'c1_metrics_{timestamp}.json')
    with open(metrics_path, 'w') as f:
        json.dump({
            'config': 'C1_zero_shot_baseline',
            'model': model_name,
            'data_path': data_path,
            'n_samples': len(samples),
            'timestamp': timestamp,
            'metrics': results,
        }, f, indent=2)
    print(f"\nSaved metrics to {metrics_path}")

    # Predictions file (for error analysis)
    preds_path = os.path.join(out_dir, f'c1_predictions_{timestamp}.json')
    with open(preds_path, 'w') as f:
        records = [{
            'id': samples[i].get('id', f'sample_{i}'),
            'stratum': strata[i],
            'question': samples[i]['qa']['question'],
            'gold': golds[i],
            'prediction': predictions[i],
        } for i in range(len(samples))]
        json.dump(records, f, indent=2)
    print(f"Saved predictions to {preds_path}")

    return results


if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--data', default='data/processed/finqa_500.json')
    parser.add_argument('--out_dir', default='results/metrics')
    parser.add_argument('--model', default='google/flan-t5-base')
    parser.add_argument('--n_samples', type=int, default=None,
                        help='Optional cap for quick testing (e.g., 50)')
    parser.add_argument('--batch_size', type=int, default=8)
    args = parser.parse_args()

    run_c1(args.data, args.out_dir, args.model, args.n_samples, args.batch_size)

Overwriting src/pipelines/c1_baseline.py


In [30]:
import os
for d in ['src', 'src/utils', 'src/evaluation', 'src/models',
          'src/pipelines', 'src/retrieval']:
    init_path = os.path.join(d, '__init__.py')
    if not os.path.exists(init_path):
        open(init_path, 'w').close()
        print(f"Created {init_path}")

In [31]:
%cd /content/drive/MyDrive/finqa-rag-lora-ablation

!python -m src.pipelines.c1_baseline --n_samples 5 --batch_size 2

/content/drive/MyDrive/finqa-rag-lora-ablation
Loading data from data/processed/finqa_500.json...
  Subsampled to 5 for quick testing
  Loaded 5 samples

Building prompts...
Loading google/flan-t5-base on cuda...
tokenizer_config.json: 2.54kB [00:00, 10.4MB/s]
spiece.model: 100% 792k/792k [00:01<00:00, 646kB/s] 
tokenizer.json: 2.42MB [00:00, 151MB/s]
special_tokens_map.json: 2.20kB [00:00, 10.5MB/s]
config.json: 1.40kB [00:00, 7.26MB/s]
model.safetensors: 100% 990M/990M [00:03<00:00, 329MB/s]
Loading weights: 100% 282/282 [00:00<00:00, 1057.55it/s, Materializing param=shared.weight]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
generation_config.json: 100% 147/147 [00:00<00:00, 864kB/s]
Model loaded. Total params: 247,577,856

Generating predictions (batch_size=2)...

Evaluating...

In [32]:
import json

# 找最新的 prediction 文件
import glob
pred_files = sorted(glob.glob('results/metrics/c1_predictions_*.json'))
latest = pred_files[-1]
print(f"Loading: {latest}\n")

with open(latest) as f:
    records = json.load(f)

# 把 5 条样本的 question / gold / prediction 都打印出来
for i, r in enumerate(records):
    print(f"--- Sample {i} ---")
    print(f"Stratum:    {r['stratum']}")
    print(f"Question:   {r['question'][:120]}")
    print(f"Gold:       {r['gold']!r}")
    print(f"Prediction: {r['prediction']!r}")
    print()

Loading: results/metrics/c1_predictions_20260504_115119.json

--- Sample 0 ---
Stratum:    percentage_simple
Question:   what percentage of the class b preferred stock is currently outstanding?
Gold:       '0%'
Prediction: 'no'

--- Sample 1 ---
Stratum:    percentage_simple
Question:   in 2015 what was the percent of the total second generation capital expenditures by type of expenditure that wassecond g
Gold:       '79.5%'
Prediction: '1'

--- Sample 2 ---
Stratum:    percentage_simple
Question:   what percentage of total consolidated revenues was gfs segment in 2016?
Gold:       '46%'
Prediction: 'no'

--- Sample 3 ---
Stratum:    percentage_simple
Question:   what percentage of total cash and investments as of dec . 28 2013 was comprised of available-for-sale investments?
Gold:       '57%'
Prediction: 'percentage'

--- Sample 4 ---
Stratum:    percentage_simple
Question:   what percentage of cash , cash equivalents , and short-term investments was due to cash generated from operati

In [33]:
%cd /content/drive/MyDrive/finqa-rag-lora-ablation

# 跑全量 500 条 (不加 --n_samples 就是全部)
!python -m src.pipelines.c1_baseline --batch_size 16

/content/drive/MyDrive/finqa-rag-lora-ablation
Loading data from data/processed/finqa_500.json...
  Loaded 500 samples

Building prompts...
Loading google/flan-t5-base on cuda...
Loading weights: 100% 282/282 [00:00<00:00, 892.13it/s, Materializing param=shared.weight]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Model loaded. Total params: 247,577,856

Generating predictions (batch_size=16)...

Evaluating...

=== C1 Baseline Results ===
Overall (500 samples):
  F1:      0.0367
  ROUGE-L: 0.0367

Per-stratum:
  boolean_complex            n= 10  F1=0.5000  ROUGE-L=0.5000
  boolean_simple             n= 40  F1=0.3000  ROUGE-L=0.3000
  numeric_complex            n= 60  F1=0.0000  ROUGE-L=0.0000
  numeric_simple             n=130  F1=0.0000  ROUGE-L=0.0000
  percentage_complex         

In [34]:
import json
import glob

# 找最新的 metrics 文件
metrics_files = sorted(glob.glob('results/metrics/c1_metrics_*.json'))
with open(metrics_files[-1]) as f:
    data = json.load(f)

print("=== C1 Baseline (full 500 samples) ===\n")
print(f"Model: {data['model']}")
print(f"Overall F1:      {data['metrics']['overall']['f1']:.4f}")
print(f"Overall ROUGE-L: {data['metrics']['overall']['rouge_l']:.4f}")

print(f"\n{'Stratum':<25} {'n':>4} {'F1':>8} {'ROUGE-L':>8}")
print("-" * 50)
for stratum in sorted(data['metrics']['by_stratum'].keys()):
    m = data['metrics']['by_stratum'][stratum]
    print(f"{stratum:<25} {m['n_samples']:>4} {m['f1']:>8.4f} {m['rouge_l']:>8.4f}")

=== C1 Baseline (full 500 samples) ===

Model: google/flan-t5-base
Overall F1:      0.0367
Overall ROUGE-L: 0.0367

Stratum                      n       F1  ROUGE-L
--------------------------------------------------
boolean_complex             10   0.5000   0.5000
boolean_simple              40   0.3000   0.3000
numeric_complex             60   0.0000   0.0000
numeric_simple             130   0.0000   0.0000
percentage_complex         130   0.0051   0.0051
percentage_simple          130   0.0051   0.0051


In [35]:
# 抽 3 条 percentage_simple, 3 条 numeric_simple, 全部 boolean 看一下
import json
import glob

pred_files = sorted(glob.glob('results/metrics/c1_predictions_*.json'))
with open(pred_files[-1]) as f:
    records = json.load(f)

from collections import defaultdict
by_stratum = defaultdict(list)
for r in records:
    by_stratum[r['stratum']].append(r)

for stratum in ['percentage_simple', 'numeric_simple', 'boolean_simple', 'boolean_complex']:
    print(f"\n========== {stratum} (showing 3) ==========")
    for r in by_stratum[stratum][:3]:
        print(f"  Q: {r['question'][:80]}")
        print(f"  Gold: {r['gold']!r}  |  Pred: {r['prediction']!r}")
        print()


========== percentage_simple (showing 3) ==========
  Q: what percentage of the class b preferred stock is currently outstanding?
  Gold: '0%'  |  Pred: 'no'

  Q: in 2015 what was the percent of the total second generation capital expenditures
  Gold: '79.5%'  |  Pred: '1'

  Q: what percentage of total consolidated revenues was gfs segment in 2016?
  Gold: '46%'  |  Pred: 'no'


========== numeric_simple (showing 3) ==========
  Q: what is the net change in net revenue during 2015?
  Gold: '-558'  |  Pred: 'percentage'

  Q: what is the average number of shares per registered holder as of february 19 , 2
  Gold: '2666022'  |  Pred: 'percentage'

  Q: considering the final year of the investment , what was the highest return for t
  Gold: '89.19'  |  Pred: 'no'


========== boolean_simple (showing 3) ==========
  Q: was the caribbean segment revenue increase greater than the south american growt
  Gold: 'yes'  |  Pred: 'yes'

  Q: did contract services expense increase more in 2012 t

In [36]:
%cd /content/drive/MyDrive/finqa-rag-lora-ablation

!git add src/
!git add results/metrics/

!git status   # 确认要提交的文件

/content/drive/MyDrive/finqa-rag-lora-ablation
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   results/metrics/c1_metrics_20260504_114932.json
	new file:   results/metrics/c1_metrics_20260504_115119.json
	new file:   results/metrics/c1_metrics_20260504_115248.json
	new file:   results/metrics/c1_predictions_20260504_114932.json
	new file:   results/metrics/c1_predictions_20260504_115119.json
	new file:   results/metrics/c1_predictions_20260504_115248.json
	new file:   src/__init__.py
	new file:   src/evaluation/__init__.py
	new file:   src/evaluation/metrics.py
	new file:   src/models/__init__.py
	new file:   src/models/baseline.py
	new file:   src/pipelines/__init__.py
	new file:   src/pipelines/c1_baseline.py
	new file:   src/retrieval/__init__.py
	new file:   src/utils/__init__.py
	new file:   src/utils/data_utils.py

Changes not staged for commit:
  (use "git add <file>..." to up

In [37]:
!git commit -m "Add C1 zero-shot baseline pipeline with Flan-T5-Base, F1/ROUGE-L evaluation, and initial 500-sample results"
!git push origin main

[main 9efb24d] Add C1 zero-shot baseline pipeline with Flan-T5-Base, F1/ROUGE-L evaluation, and initial 500-sample results
 16 files changed, 4040 insertions(+)
 create mode 100644 results/metrics/c1_metrics_20260504_114932.json
 create mode 100644 results/metrics/c1_metrics_20260504_115119.json
 create mode 100644 results/metrics/c1_metrics_20260504_115248.json
 create mode 100644 results/metrics/c1_predictions_20260504_114932.json
 create mode 100644 results/metrics/c1_predictions_20260504_115119.json
 create mode 100644 results/metrics/c1_predictions_20260504_115248.json
 create mode 100644 src/__init__.py
 create mode 100644 src/evaluation/__init__.py
 create mode 100644 src/evaluation/metrics.py
 create mode 100644 src/models/__init__.py
 create mode 100644 src/models/baseline.py
 create mode 100644 src/pipelines/__init__.py
 create mode 100644 src/pipelines/c1_baseline.py
 create mode 100644 src/retrieval/__init__.py
 create mode 100644 src/utils/__init__.py
 create mode 100644 s

In [43]:
print("=== .gitignore 完整内容 ===")
!cat .gitignore

=== .gitignore 完整内容 ===
# Byte-compiled / optimized / DLL files
__pycache__/
*.py[codz]
*$py.class

# C extensions
*.so

# Distribution / packaging
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
share/python-wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST

# PyInstaller
#   Usually these files are written by a python script from a template
#   before PyInstaller builds the exe, so as to inject date/other infos into it.
*.manifest
*.spec

# Installer logs
pip-log.txt
pip-delete-this-directory.txt

# Unit test / coverage reports
htmlcov/
.tox/
.nox/
.coverage
.coverage.*
.cache
nosetests.xml
coverage.xml
*.cover
*.py.cover
.hypothesis/
.pytest_cache/
cover/

# Translations
*.mo
*.pot

# Django stuff:
*.log
local_settings.py
db.sqlite3
db.sqlite3-journal

# Flask stuff:
instance/
.webassets-cache

# Scrapy stuff:
.scrapy

# Sphinx documentation
docs/_build/

# PyBuilder
.pybuilder/
target/

# Jupyter Notebook
.ipynb_checkpoints

# IPyth

In [44]:
# 看 working directory 是否正确
import os
print("CWD:", os.getcwd())

# 看 git 仓库根
!git rev-parse --show-toplevel

# 看 git 内部状态
!git status -uall   # -u 显示所有 untracked 文件

CWD: /content/drive/MyDrive/finqa-rag-lora-ablation
/content/drive/MyDrive/finqa-rag-lora-ablation
On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   notebooks/01_data_exploration.ipynb

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/01_data_exploration.ipynb

